# Data cleaning

In [ ]:
from typing import Optional, Dict, Any
from collections import Counter
import re
import numpy as np
from rdkit import Chem
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
import pandas as pd
from rdkit.Chem import rdmolops

In [ ]:
"""
Fix Ru complex SMILES in cleaned_SMILES.csv by fragmenting
into separate ligands for proper aromaticity perception.

Generates the replacement dictionary automatically, then applies it.
"""
# ── Load data ──
df = pd.read_csv("Data/cleaned_SMILES.csv", index_col=0)
print(f"Total entries: {len(df)}")

# ── Hardcoded corrections ──
# Each Ru complex → its ligands as dot-separated fragments
# with proper aromatic pyridine rings

RU_CORRECTIONS = {
    # N749 variant (74 entries) — terpyridine with C=C bonds
    "S=C=N[Ru]12(N=C=S)(N3C=C[C@H](C=C3C3=C[C@@H](C=CN23)C(=O)O)C(=O)O)N2C=C[C@H](C=C2C2=C[C@@H](C=CN12)C(=O)O)C(=O)O":
    "[Ru].S=C=N.S=C=N.O=C(O)c1ccnc(-c2cc(C(=O)O)ccn2)c1.O=C(O)c1ccnc(-c2cc(C(=O)O)ccn2)c1",

    # N749 / Black Dye (331 entries) — terpyridine, fully saturated in original
    "O=C([O-])C1CCN2C(C1)C1CC(C(=O)O)CCN1[Ru]21(N=C=S)(N=C=S)N2CCC(C(=O)[O-])CC2C2CC(C(=O)O)CCN21":
    "[Ru].S=C=N.S=C=N.O=C([O-])c1ccnc(-c2cc(C(=O)O)ccn2)c1.O=C([O-])c1ccnc(-c2cc(C(=O)O)ccn2)c1",

    # Z907 (55 entries) — bipyridine with alkyl chains
    "CCCCCCCCCC1CCN2C(C1)C1CC(CCCCCCCCC)CCN1[Ru]12(N=C=S)(N=C=S)N2CCC(CC2C2N1CCC(C2)C(=O)O)C(=O)O":
    "[Ru].S=C=N.S=C=N.CCCCCCCCCCc1ccnc(-c2cc(CCCCCCCCC)ccn2)c1.O=C(O)c1ccnc(-c2cc(C(=O)O)ccn2)c1",

    # N749 with 3 NCS (14 entries)
    "O=C([O-])C1CCN2C(C1)C1CC(C(=O)[O-])CC3C4CC(C(=O)[O-])CCN4[Ru]2(N=C=S)(N=C=S)(N=C=S)N13":
    "[Ru].S=C=N.S=C=N.S=C=N.O=C([O-])c1ccnc(-c2cc(C(=O)[O-])cc(-c3cc(C(=O)[O-])ccn3)n2)c1",

    # N749 variant with counterions removed (1 entry)
    "O=C([O-])[C@@H]1C=CN2C(=C1)C1=C[C@H](C(=O)O)C=CN1[Ru]21(N=C=S)(N=C=S)N2C=C[C@@H](C(=O)[O-])C=C2C2=C[C@H](C(=O)O)C=CN21":
    "[Ru].S=C=N.S=C=N.O=C([O-])c1ccnc(-c2cc(C(=O)O)ccn2)c1.O=C(O)c1ccnc(-c2cc(C(=O)[O-])ccn2)c1",

    # Z907 variant with long chains (3 entries)
    "CCCCCCCCCCCCCC1C=CN2C(=C1)C1=CC(CCCCCCCCCCCCC)C=CN1[Ru]12(N=C=S)(N=C=S)N2C=C[C@H](C=C2C2=C[C@@H](C=CN12)C(=O)O)C(=O)O":
    "[Ru].S=C=N.S=C=N.CCCCCCCCCCCCCCc1ccnc(-c2cc(CCCCCCCCCCCCC)ccn2)c1.O=C(O)c1ccnc(-c2cc(C(=O)O)ccn2)c1",
}

gen = GetMorganGenerator(radius=3, fpSize=1024)

print("Validating corrections...")
for old, new in RU_CORRECTIONS.items():
    mol_old = Chem.MolFromSmiles(old)
    mol_new = Chem.MolFromSmiles(new)

    if mol_new is None:
        print(f"  PARSE FAILED: {new[:70]}")
        continue

    na_old = sum(1 for a in mol_old.GetAtoms() if a.GetIsAromatic())
    na_new = sum(1 for a in mol_new.GetAtoms() if a.GetIsAromatic())

    fp_old = gen.GetFingerprint(mol_old)
    fp_new = gen.GetFingerprint(mol_new)
    diff = sum(1 for i in range(1024) if fp_old[i] != fp_new[i])

    n = (df['SMILES'] == old).sum()
    print(f"  [{n:3d} entries] Aromatic: {na_old} → {na_new}, Bits changed: {diff}")

# Apply
df['SMILES'] = df['SMILES'].replace(RU_CORRECTIONS)
df.to_csv("Data/cleaned_SMILES_fixed.csv")
print("\nSaved to Data/cleaned_SMILES_fixed.csv")

Total entries: 4426
Validating corrections...
  [ 74 entries] Aromatic: 0 → 24, Bits changed: 55
  [331 entries] Aromatic: 0 → 24, Bits changed: 64
  [ 55 entries] Aromatic: 0 → 24, Bits changed: 79
  [ 14 entries] Aromatic: 0 → 18, Bits changed: 69
  [  1 entries] Aromatic: 0 → 24, Bits changed: 62
  [  3 entries] Aromatic: 0 → 24, Bits changed: 78

Saved to Data/cleaned_SMILES_fixed.csv


In [33]:
def extract_redox_couple(i):
    s = str(i)

    # --- Specific / rare couples first ---
    
    # Copper complexes
    if re.search(r'\bCu[I]{1,2}\b|CuI|CuII|\bcopper\b', s, re.IGNORECASE):
        return 'Cu(I)/Cu(II)'
    
    # TEMPO radical mediator
    if re.search(r'TEMPO|NOBF4|nitrox', s, re.IGNORECASE):
        return 'TEMPO/Iodide'
    
    # Cobalt complexes — check before iodide since Co electrolytes often contain LiI as additive
    if re.search(r'\bcobalt\b|\bCo\s*\(', s, re.IGNORECASE):
        return 'Co(II)/Co(III)'
    # Also catch shorthand like Co-bpy, Co-phen, [Co(bpy)
    if re.search(r'Co-bpy|Co-phen|Co-dmbpy|\[Co\(', s, re.IGNORECASE):
        return 'Co(II)/Co(III)'
    
    # Bromide/Tribromide — use word boundary or context, not bare "Br"
    if re.search(r'\bbromide\b|\btribromide\b|DMBIBr|BMImBr|TPABr|\bBr2\b', s, re.IGNORECASE):
        return 'Bromide/Tribromide'
    
    # Spiro-OMeTAD solid-state HTM
    if re.search(r'spiro|MeOTAD|OmeTAD', s, re.IGNORECASE):
        return 'Spiro-OMeTAD'
    
    # DMPIC/DMPIDC couple (chloride-based imidazolium)
    if re.search(r'DMPIC|DMPIDC', s):
        return 'DMPIC/DMPIDC'
    
    # Thiocyanate/selenocyanate systems
    if re.search(r'EMISCN|SeCN|PyC6|Py2C6', s):
        return 'SCN-/SeCN-'
    
    # Solid-state (generic)
    if re.search(r'solid\s*(state|organic)', s, re.IGNORECASE):
        return 'Solid_HTM'
    
    # Commercial iodide products — map to iodide
    if re.search(r'Iodolyte|Solaronix|Mosalyte|EL-HSE|EL141|HI-30|AN-50|Z-50|HPE|DHS-Z23|Dyesol|Heptachroma', s, re.IGNORECASE):
        return 'Iodide/Triiodide'
    
    # Iodide/Triiodide — check last as components appear as additives elsewhere
    if re.search(r'\biodide\b|\btriiodide\b|\bI2\b|\bLiI\b|\bNaI\b|\bKI\b'
                 r'|DMPImI|DMPII|PMII|BMII|DMII|EMII|TBAI|TPAI|HMII|DMHII'
                 r'|\biodine\b|OPV-AN-I', s, re.IGNORECASE):
        return 'Iodide/Triiodide'
    
    # Truly unidentifiable
    return np.nan

def extract_I2_conc(s, redox_couple):
    """
    Extract I2 concentration from electrolyte string.
    Returns 0 if redox couple is not Iodide/Triiodide.
    Returns np.nan if iodide-based but concentration not found.
    """
    if redox_couple != 'Iodide/Triiodide':
        return 0.0
    
    match = re.search(
        r'([\d.]+)\s*[mM]{0,1}\s*(?:M\s*)?'   # concentration + unit
        r'(?:I2\b|iodine\b|I₂)',               # I2 identifiers
        str(s), re.IGNORECASE
    )
    return float(match.group(1)) if match else np.nan

# ---------- compiled regexes for SEMICONDUCTOR ----------
UNIT_GROUP = r'(?:u\s*m|u?m|µm|mm)'

# strip nm tokens/ranges (keep rest)
RE_STRIP_NM = re.compile(
    r'\b\d+(?:\.\d+)?\s*(?:[-–]\s*\d+(?:\.\d+)?)?\s*nm\b'
    r'(?:\s*(?:particles?|particle\s*size))?',
    re.IGNORECASE
)

# material type (first token)
RE_TYPE = re.compile(r'^\s*(?P<type>[A-Za-z][A-Za-z0-9_+\-]*)', re.IGNORECASE)

# --- edge case A: explicit pair with words
RE_PAIR_WITH_WORDS_FILM_FIRST = re.compile(
    rf'\(\s*'
    rf'(?P<film>\d+(?:\.\d+)?)\s*(?P<film_unit>{UNIT_GROUP})\s*transp\w*[^()+]*\+\s*'
    rf'(?P<scat>\d+(?:\.\d+)?)\s*(?P<scat_unit>{UNIT_GROUP})\s*scatt\w*(?:\s*layer)?'
    rf'\s*\)',
    re.IGNORECASE
)

RE_PAIR_WITH_WORDS_SCAT_FIRST = re.compile(
    rf'\(\s*'
    rf'(?P<scat>\d+(?:\.\d+)?)\s*(?P<scat_unit>{UNIT_GROUP})\s*scatt\w*(?:\s*layer)?[^()+]*\+\s*'
    rf'(?P<film>\d+(?:\.\d+)?)\s*(?P<film_unit>{UNIT_GROUP})\s*transp\w*'
    rf'\s*\)',
    re.IGNORECASE
)

# --- edge case B: shared-unit pair like "(12 + 4 um thick)" ---
RE_PAIR_SHARED_UNIT = re.compile(
    rf'\(\s*(?P<film>\d+(?:\.\d+)?)\s*\+\s*(?P<scat>\d+(?:\.\d+)?)\s*(?P<unit>{UNIT_GROUP})\b[^)]*\)',
    re.IGNORECASE
)

# Pair with single unit after second number (requires '+' or 'and'):
RE_PAIR_BOTH_IN_PARENS = re.compile(
    rf'\(\s*(?P<film>\d+(?:\.\d+)?)\s*(?:{UNIT_GROUP})?\s*(?:\+|and)\s*'
    rf'(?P<scat>\d+(?:\.\d+)?)\s*(?P<unit>{UNIT_GROUP})\s*\)',
    re.IGNORECASE
)

# Film range like "(16-18 um)" or "(16–18 um)"
RE_FILM_RANGE = re.compile(
    rf'\(\s*(?P<val1>\d+(?:\.\d+)?)\s*[-–]\s*(?P<val2>\d+(?:\.\d+)?)\s*(?P<unit>{UNIT_GROUP})\s*\)',
    re.IGNORECASE
)

# Scattering thickness (order matters: try TEXT_FIRST, then NUM_FIRST)
RE_SCAT_TEXT_FIRST = re.compile(
    rf'\bscatt\w*(?:\s*layer)?(?:(?!\d).){{0,40}}(?P<val>\d+(?:\.\d+)?)\s*(?P<unit>{UNIT_GROUP})',
    re.IGNORECASE
)
RE_SCAT_NUM_FIRST = re.compile(
    rf'(?P<val>\d+(?:\.\d+)?)\s*(?P<unit>{UNIT_GROUP})(?:(?!\d).){{0,20}}\bscatt\w*(?:\s*layer)?',
    re.IGNORECASE
)

# Explicit film phrases
RE_FILM_IN_THICKNESS = re.compile(
    rf'\bfilm[s]?\b(?:(?!\d).){{0,80}}(?P<val>\d+(?:\.\d+)?)\s*(?P<unit>{UNIT_GROUP})\s+in\s+thickness',
    re.IGNORECASE
)
RE_FILM_THICK = re.compile(
    rf'\bfilm[s]?\b(?:(?!\d).){{0,80}}(?P<val>\d+(?:\.\d+)?)\s*(?P<unit>{UNIT_GROUP})\s*thick',
    re.IGNORECASE
)

# Fallbacks
RE_NUM_UNIT = re.compile(rf'(?P<val>\d+(?:\.\d+)?)\s*(?P<unit>{UNIT_GROUP})', re.IGNORECASE)
RE_FILM_WORD = re.compile(r'\b(?:thin\s+film[s]?|film[s]?)\b', re.IGNORECASE)

def _to_um(val_str: str, unit: str = "um") -> Optional[float]:
    """Convert number + unit to micrometers (µm)."""
    try:
        v = float(val_str)
    except Exception:
        return None
    u = unit.lower().replace(" ", "")
    if u == "mm" or u.startswith("millimeter"):
        return v * 1000.0
    # treat um/u m/uM/µm as micrometers
    return v

def parse_semiconductor(s: str) -> pd.Series:
    out = {
        "Semiconductor_type":  None, 
        "Semiconductor_thick": None, 
        "Scatt":               None
    }
    
    if not isinstance(s, str) or not s.strip():
        return pd.Series(out)

    txt = s.strip()

    # Type
    mtype = RE_TYPE.search(txt)
    if mtype:
        out["Semiconductor_type"] = mtype.group("type")

    # Remove nm tokens/ranges; normalize spaces & soft hyphen
    cleaned = RE_STRIP_NM.sub("", txt)
    cleaned = cleaned.replace("\u00ad", "")
    cleaned = re.sub(r'\s{2,}', ' ', cleaned).strip()

    # --- Film range
    mrange = RE_FILM_RANGE.search(cleaned)
    if mrange:
        v1 = _to_um(mrange.group("val1"), mrange.group("unit"))
        v2 = _to_um(mrange.group("val2"), mrange.group("unit"))
        if v1 is not None and v2 is not None:
            out["Semiconductor_thick"] = (v1 + v2) / 2.0
        return pd.Series(out)

    # --- Pair with words
    mwords = RE_PAIR_WITH_WORDS_FILM_FIRST.search(cleaned) or RE_PAIR_WITH_WORDS_SCAT_FIRST.search(cleaned)
    if mwords:
        out["Semiconductor_thick"] = _to_um(mwords.group("film"), mwords.group("film_unit"))
        out["Scatt"]               = _to_um(mwords.group("scat"), mwords.group("scat_unit"))
        return pd.Series(out)

    # --- Shared-unit pair
    mshared = RE_PAIR_SHARED_UNIT.search(cleaned)
    if mshared:
        out["Semiconductor_thick"] = _to_um(mshared.group("film"), mshared.group("unit"))
        out["Scatt"]               = _to_um(mshared.group("scat"), mshared.group("unit"))
        return pd.Series(out)

    # --- Simple pair
    mpair = RE_PAIR_BOTH_IN_PARENS.search(cleaned)
    if mpair:
        unit = mpair.group("unit")
        out["Semiconductor_thick"] = _to_um(mpair.group("film"), unit)
        out["Scatt"]               = _to_um(mpair.group("scat"), unit)
        return pd.Series(out)

    # --- Scattering first
    msc = RE_SCAT_TEXT_FIRST.search(cleaned) or RE_SCAT_NUM_FIRST.search(cleaned)
    scat_span = None
    if msc:
        out["Scatt"] = _to_um(msc.group("val"), msc.group("unit"))
        scat_span = msc.span()

    # --- Film explicit
    mfilm = RE_FILM_IN_THICKNESS.search(cleaned) or RE_FILM_THICK.search(cleaned)
    if mfilm:
        out["Semiconductor_thick"] = _to_um(mfilm.group("val"), mfilm.group("unit"))
        return pd.Series(out)

    # --- Film fallback
    film_anchor = RE_FILM_WORD.search(cleaned)
    film_start = film_anchor.start() if film_anchor else -1
    for m in RE_NUM_UNIT.finditer(cleaned):
        if scat_span and (m.start() >= scat_span[0] and m.end() <= scat_span[1]):
            continue
        if film_start == -1 or m.start() >= film_start:
            out["Semiconductor_thick"] = _to_um(m.group("val"), m.group("unit"))
            break

    return pd.Series(out)

In [34]:
# Updated patterns based on actual data
_mwcm2_re = re.compile(
    r'(?P<val>\d+(?:\.\d+)?)'   # number
    r'\s*'                       # optional space
    r'm[wW]\s*/\s*cm2'          # mW/cm2 or mw/cm2, spaces around /
    , re.IGNORECASE              # catches mW, mw, MW
)

_sun_pct_re = re.compile(
    r'(?P<pct>\d+(?:\.\d+)?)\s*%\s*sun', 
    re.IGNORECASE
)

_sun_frac_re = re.compile(
    r'(?P<frac>\d+(?:\.\d+)?)\s*sun', 
    re.IGNORECASE
)

_am_tag_re = re.compile(
    r'\bam\s*1\.?\s*5\s*[Gg]?\b',   # AM1.5, AM 1.5, AM 1.5G, AM 1.5 G
    re.IGNORECASE
)

def parse_solar_simulator(cell, one_sun_mwcm2: float = 100.0):
    if pd.isna(cell):
        return np.nan

    s = str(cell).strip()
    if s in {"", "-"}:
        return np.nan

    t = re.sub(r'\s{2,}', ' ', s).strip()

    # 1) Direct mW/cm2
    m = _mwcm2_re.search(t)
    if m:
        return float(m.group("val"))

    # 2) Percent sun
    m = _sun_pct_re.search(t)
    if m:
        return (float(m.group("pct")) / 100.0) * one_sun_mwcm2

    # 3) Fractional sun — before AM tag check, but exclude integers >= 1
    #    to avoid matching "1.5" in bare "AM 1.5G" strings
    m = _sun_frac_re.search(t)
    if m:
        val = float(m.group("frac"))
        # Only accept if it looks like a deliberate irradiance value
        # i.e. not just the "1.5" from the AM tag itself
        am_match = _am_tag_re.search(t)
        if am_match:
            # Check the sun match isn't overlapping with the AM tag
            sun_start = m.start()
            am_end = am_match.end()
            if sun_start > am_end:          # sun value appears after the AM tag
                return val * one_sun_mwcm2
        else:
            return val * one_sun_mwcm2      # no AM tag, safe to use

    # 4) AM tag with no irradiance → nan
    if _am_tag_re.search(t):
        return np.nan

    return np.nan

NA = 6.02214076e23  # Avogadro constant
UNIT_FACTORS = {
    "nmol/cm2": 1.0,
    "mol/cm2": 1e9,  # mol → nmol
    "mmol/cm3": None,  # Can't convert to cm² without thickness
    "nmol/mm3": None,  # Same issue as mmol/cm³
    "molecules/cm2": 1e9 / NA,  # molecules → nmol
    "mols/mm": None  # no direct surface conversion
}

def parse_dye_loading(cell):
    if pd.isna(cell):
        return np.nan
    
    s = str(cell).strip()
    if s in {"", "-"}:
        return np.nan
    
    # Replace multiple spaces and commas
    s = re.sub(r"[ ,]+", " ", s)
    
    # Handle sums like "95 nmol/cm2 + 52 nmol/cm2"
    parts = re.split(r"\s*\+\s*", s)
    total_nmol_cm2 = 0.0
    parsed_any = False
    
    for part in parts:
        # Scientific notation with optional units
        m = re.match(r"([0-9]*\.?[0-9]+(?:e[+-]?[0-9]+)?)\s*([a-zA-Z+/0-9.]+)", part)
        if not m:
            continue
        
        val = float(m.group(1))
        unit = m.group(2).lower()
        
        # Normalize weird spacing, e.g., 'nmol/cm2' vs 'nmol / cm2'
        unit = unit.replace(" ", "")
        
        # Match known units
        if "nmol/cm2" in unit:
            total_nmol_cm2 += val
            parsed_any = True
        elif "mol/cm2" in unit and not "nmol" in unit:
            total_nmol_cm2 += val * 1e9
            parsed_any = True
        elif "molecules/cm2" in unit:
            total_nmol_cm2 += val * (1e9 / NA)
            parsed_any = True
        # If mmol/cm³ or nmol/mm³, skip because we lack thickness
        elif "mmol/cm3" in unit or "nmol/mm3" in unit or "mols/mm" in unit:
            return np.nan
    
    return total_nmol_cm2 if parsed_any else np.nan

def parse_exposure_time(s):
    if pd.isna(s) or str(s).strip() in ["-", ""]:
        return np.nan
    
    text = str(s).lower().strip()
    
    # Replace slashes with hyphen for ranges
    text = text.replace("/", "-")
    
    def parse_single_value(val):
        val = val.strip()
        # minutes to hours
        m = re.match(r"(\d+(?:\.\d+)?)\s*(min|minutes?)", val)
        if m:
            return float(m.group(1)) / 60.0
        # hours
        m = re.match(r"(\d+(?:\.\d+)?)\s*(h|hours?|hour)", val)
        if m:
            return float(m.group(1))
        return None
    
    # Handle multi-part exposures separated by "+"
    if "+" in text:
        parts = text.split("+")
        total = 0.0
        for part in parts:
            # Extract all numeric+unit patterns from each part
            matches = re.findall(r"(\d+(?:\.\d+)?)\s*(h|hours?|hour|min|minutes?)", part)
            for num, unit in matches:
                num = float(num)
                if unit.startswith("min"):
                    num /= 60.0
                total += num
        return total
    
    # Handle ranges (e.g., "16-17 hours" or "8-12 h")
    range_match = re.match(r"(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)\s*(h|hours?|hour|min|minutes?)", text)
    if range_match:
        v1, v2, unit = range_match.groups()
        v1, v2 = float(v1), float(v2)
        if unit.startswith("min"):
            v1 /= 60.0
            v2 /= 60.0
        return (v1 + v2) / 2.0
    
    # Otherwise, try direct parsing
    val = parse_single_value(text)
    if val is not None:
        return val
    
    # If no direct match, look for any number-unit pairs and sum
    matches = re.findall(r"(\d+(?:\.\d+)?)\s*(h|hours?|hour|min|minutes?)", text)
    if matches:
        total = 0.0
        for num, unit in matches:
            num = float(num)
            if unit.startswith("min"):
                num /= 60.0
            total += num
        return total if total > 0 else np.nan
    
    return np.nan

_NUM = r'[-+]?\d+(?:[.,]\d+)?(?:e[+\-]?\d+)?'  # supports 1, 1.2, 1,2, 1e-3, 1.2e+3

def _to_float(x: str) -> float:
    return float(x.replace(',', '.'))

def parse_active_area(area_str):
    # Missing or placeholder
    if pd.isna(area_str):
        return np.nan
    s = str(area_str).strip()
    if s in {"", "-"}:
        return np.nan

    s_low = s.lower()

    # --- detect unit for later conversion ---
    unit = "cm2"  # default
    if re.search(r'\bmm\s*\^?\s*2|\bmm2|mm²', s_low):
        unit = "mm2"
    elif re.search(r'\bm\s*\^?\s*2|\bm2|m²', s_low):
        unit = "m2"
    elif re.search(r'\bcm\s*\^?\s*2|\bcm2|cm²', s_low):
        unit = "cm2"

    # --- clean obvious unit tokens and parentheses to ease number matching ---
    s_clean = re.sub(r'[\(\)]', ' ', s_low)
    s_clean = re.sub(r'\b(cm|mm|m)\s*\^?\s*2|cm²|mm²|m²|cm2|mm2|m2', ' ', s_clean)
    s_clean = re.sub(r'\s{2,}', ' ', s_clean).strip()

    # --- try range first: "a-b", "a–b", or "a/b" (keep only number tokens) ---
    m_range = re.search(rf'({_NUM})\s*[-–/]\s*({_NUM})', s_clean)
    if m_range:
        a = _to_float(m_range.group(1))
        b = _to_float(m_range.group(2))
        val = (a + b) / 2.0
    else:
        # single number: take the first numeric token
        m_single = re.search(rf'({_NUM})', s_clean)
        if not m_single:
            return np.nan
        val = _to_float(m_single.group(1))

    # --- unit conversion to cm^2 ---
    if unit == "mm2":
        val = val / 100.0        # 1 cm2 = 100 mm2
    elif unit == "m2":
        val = val * 10000.0      # 1 m2 = 10,000 cm2
    # else already cm²

    return val

In [35]:
venkatraman = pd.read_csv('Data/rawDSSCDBdata.csv')
smiles_df = pd.read_csv('Data/cleaned_SMILES_fixed.csv', index_col=0)
venkatraman['Molecule SMILE'] = smiles_df['SMILES']
print(venkatraman.columns)
print('Number of entries: ', len(venkatraman))
print('Number of unique molecules:',len(venkatraman['Molecule SMILE'].unique()))

dsscdb_formatted = pd.DataFrame(data=venkatraman[['VOC', 'JSC', 'FF', 'PCE', 'Molecule SMILE', 'Electrolyte', 'Semiconductor', 'Solar simulator', 
                                                  'Exposure time', 'Dye loading', 'Active area', 'Article DOI']], copy=True)
dsscdb_formatted['Redox_couple'] = dsscdb_formatted['Electrolyte'].apply(extract_redox_couple)
dsscdb_formatted['I2_conc_M'] = dsscdb_formatted.apply(
    lambda row: extract_I2_conc(row['Electrolyte'], row['Redox_couple']), 
    axis=1
)
dsscdb_formatted[['Semiconductor_type', 'Semiconductor_thick', 'Scatt']] = dsscdb_formatted['Semiconductor'].apply(parse_semiconductor)
dsscdb_formatted['Solar_simulator'] = dsscdb_formatted['Solar simulator'].apply(parse_solar_simulator)
dsscdb_formatted['Dye_loading'] = dsscdb_formatted['Dye loading'].apply(parse_dye_loading)
dsscdb_formatted['Exposure_time'] = dsscdb_formatted['Exposure time'].apply(parse_exposure_time)
dsscdb_formatted['Active_area'] = dsscdb_formatted['Active area'].apply(parse_active_area)
dsscdb_formatted = dsscdb_formatted.drop(columns=['Electrolyte', 'Semiconductor', 'Solar simulator', 
                                                  'Exposure time', 'Dye loading', 'Active area'])
dsscdb_formatted.rename(columns={'Molecule SMILE': 'SMILES', 'Article DOI': 'DOI'}, inplace=True)
dsscdb_formatted

Index(['VOC', 'JSC', 'FF', 'PCE', 'Electrolyte', 'Active area', 'Co-adsorbent',
       'Co-sensitizer', 'Semiconductor', 'Dye loading', 'Exposure time',
       'Solar simulator', 'Performance comment', 'Article author',
       'Article title', 'Article journal', 'Article volume', 'Article DOI',
       'Article pages', 'Article issue nr', 'Article EID', 'Article year',
       'Article year.1', 'Article electronic id', 'Article keywords',
       'Molecule SMILE', 'Molecule keywords',
       'Molecule spectrum absorption maxima',
       'Molecule spectrum emission maxima', 'Molecule spectrum solvent'],
      dtype='object')
Number of entries:  4426
Number of unique molecules: 2367


,VOC,JSC,FF,PCE,SMILES,DOI,Redox_couple,I2_conc_M,Semiconductor_type,Semiconductor_thick,Scatt,Solar_simulator,Dye_loading,Exposure_time,Active_area
0,687.0,10.79,0.70,5.19,N#C/C(=C\\c1ccc(cc1)N(c1ccccc1)CCCCCCN1c2ccccc...,10.1016/j.dyepig.2012.02.011,Iodide/Triiodide,0.03,TiO2,NaN,NaN,100.0,NaN,15.0,NaN
1,790.0,6.90,0.47,2.60,N#C/C(=C\\c1ccc(s1)c1ccc(s1)c1ccc(cc1)N(c1ccc(...,10.1039/c0ee00218f,Spiro-OMeTAD,0.00,TiO2,8.0,5.0,100.0,NaN,5.0,0.158
2,835.0,7.70,0.49,3.10,N#C/C(=C\\c1ccc(s1)c1ccc(cc1)N(c1ccc(cc1)/C=C/...,10.1039/c0ee00218f,Spiro-OMeTAD,0.00,TiO2,8.0,5.0,100.0,NaN,5.0,0.158
3,800.0,6.40,0.43,2.20,CCN(c1ccc2c(c1)oc(=O)c(c2)/C=C/c1ccc(cc1)N(c1c...,10.1039/c0ee00218f,Spiro-OMeTAD,0.00,TiO2,8.0,5.0,100.0,NaN,5.0,0.158
4,651.0,10.30,0.75,5.00,N#C/C(=C\\c1ccc(s1)c1ccc(s1)c1ccc(cc1)N(c1ccc(...,10.1039/c0ee00218f,Iodide/Triiodide,NaN,TiO2,8.0,5.0,100.0,NaN,5.0,0.158
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4421,666.0,17.84,0.74,8.79,[Ru].S=C=N.S=C=N.O=C([O-])c1ccnc(-c2cc(C(=O)O)...,10.1016/j.dyepig.2018.03.075,Iodide/Triiodide,0.02,TiO2,14.0,NaN,100.0,NaN,24.0,0.120
4422,651.0,9.93,0.70,4.52,C(#N)/C(/C(=O)O)=C\\C=1SC(=CC1)C=1C=CC=2N(C3=C...,10.1016/j.dyepig.2018.03.075,Iodide/Triiodide,0.02,TiO2,14.0,NaN,100.0,296.0,24.0,0.120
4423,667.0,12.85,0.71,6.09,CSC1=CC=C(C=C1)C=1C=C2SC=3C=C(C=CC3N(C2=CC1)CC...,10.1016/j.dyepig.2018.03.075,Iodide/Triiodide,0.02,TiO2,14.0,NaN,100.0,224.0,24.0,0.120
4424,659.0,12.46,0.69,5.65,COC1=CC=C(C=C1)C=1C=C2SC=3C=C(C=CC3N(C2=CC1)CC...,10.1016/j.dyepig.2018.03.075,Iodide/Triiodide,0.02,TiO2,14.0,NaN,100.0,187.0,24.0,0.120


In [36]:
# ── LOOKUP TABLES & FUNCTIONS ─────────────────────────────────────
COADSORBENT_EXACT = {
    'chenodeoxycholic acid':                              'CDCA',
    '3a,7a-dihydroxy-5b-cholic acid':                    'CDCA',
    'tetrabutylammonium Deoxycholate':                    'DCA',
    'tetrabutylammonium deoxycholate':                    'DCA',
    'tetrabutylammonium\r\nDeoxycholate':                 'DCA',
    'tetrabutylammonium \nDeoxycholate':                  'DCA',
    'deoxycholate':                                       'DCA',
    '4-guanidino\r\nbutyric acid':                        '4-Guanidinobutyric acid',
    '1-decylphosphonic Acid':                             '1-Decylphosphonic acid',
    'HC-A':                                               'HC-A1',
    'CCCC[C@@H](COc1ccc(cc1)c1ccc2c(c1)Sc1c(N2c2ccc(cc2)C(=O)O)ccc(c1)c1ccc(cc1)OC[C@H](CCCC)CC)CC':    'PTZ1',
    'CCCC[C@@H](COc1ccc(cc1)c1ccc2c(c1)Sc1c(N2[C@@H]2CC=C(C=C2)c2ccc(cc2)C(=O)O)ccc(c1)c1ccc(cc1)OC[C@H](CCCC)CC)CC': 'PTZ2',
    'CC1(C2=CC=CC=C2C=2C=CC(=CC12)N(C1=CC=C(C(=O)O)C=C1)C1=CC=2C(C3=CC=CC=C3C2C=C1)(C)C)C':             'HC-A4',
    'C(C)C(COC1=CC=C(C=C1)C=1C=CC=2N(C3=CC=C(C=C3C2C1)C1=CC=C(C=C1)OCC(CCCC)CC)C1=CC=C(C(=O)O)C=C1)CCCC': 'HC-A1',
}

COADSORBENT_MAP = [
    (r'CDCA|chenodeoxycholic',                           'CDCA'),
    (r'\bDCA\b|deoxycholic',                             'DCA'),
    (r'cholic\s*acid',                                   'Cholic acid'),
    (r'cholanic\s*acid',                                 'Cholanic acid'),
    (r'isooctyl.*silane|isooctyltrieth|isooctyltrimeth', 'Isooctylsilane'),
    (r'4-guanidino.*butyric|guanidino.*butyric',         '4-Guanidinobutyric acid'),
    (r'octanoic\s*acid',                                 'Octanoic acid'),
    (r'\bDINHOP\b',                                      'DINHOP'),
    (r'\bHC-A1\b',                                       'HC-A1'),
    (r'\bHC-A4\b',                                       'HC-A4'),
    (r'\bHC-A\b(?!1|4)',                                 'HC-A1'),
    (r'diphen.*phosphinic|diphenyphosphinic',             'Diphenylphosphinic acid'),
    (r'3-phenylpropionic',                               '3-Phenylpropionic acid'),
    (r'decylphosphonic',                                 '1-Decylphosphonic acid'),
    (r'\bPTZ1\b',                                        'PTZ1'),
    (r'\bPTZ2\b',                                        'PTZ2'),
    (r'tetrabutylammonium.*deoxycholate',                'DCA'),
]

UNIT_TO_MM = {
    'mm':  1.0,
    'um':  1e-3,
    'μm':  1e-3,
    'nm':  1e-6,
}

NOISE_ENTRIES = ['TiO2 film']

COSENSITIZER_EXACT = {
    'N719, Ruthenium':   'N719',       # Ruthenium is not a co-sensitizer name
    'FNE46 cosensitizer':'FNE46',
    'H2LD14+AN-3':       'H2LD14/AN-3', # normalise separator to /
    'D149/D131':         'D149/D131',   # keep as-is — genuinely two dyes
}

# Canonical names: DCM, Chloroform, THF, Ethanol, MeOH, ACN, DMF, 
#                  Toluene, DMSO, Dioxane, Chlorobenzene, o-DCB
# Mixtures: canonical components joined by '/'

SOLVENT_MAP = {
    # ── DCM ──────────────────────────────────────────────────────────────────
    'dichloromethane':          'DCM',
    'Dichloromethane':          'DCM',
    'dichloromethane.':         'DCM',
    'DCM':                      'DCM',
    'CH2Cl2':                   'DCM',

    # ── Chloroform ───────────────────────────────────────────────────────────
    'chloroform':               'Chloroform',
    'Chloroform':               'Chloroform',
    'CHCl3':                    'Chloroform',

    # ── THF ──────────────────────────────────────────────────────────────────
    'THF':                      'THF',
    'Tetrahydrofuran':          'THF',

    # ── Ethanol ──────────────────────────────────────────────────────────────
    'ethanol':                  'Ethanol',
    'Ethanol':                  'Ethanol',
    'EtOH':                     'Ethanol',

    # ── Methanol ─────────────────────────────────────────────────────────────
    'methanol':                 'MeOH',
    'CH3OH':                    'MeOH',
    'MeOH':                     'MeOH',

    # ── Acetonitrile ─────────────────────────────────────────────────────────
    'acetonitrile':             'ACN',
    'Acetonitrile':             'ACN',
    'CH3CN':                    'ACN',
    'MeCN':                     'ACN',

    # ── DMF ──────────────────────────────────────────────────────────────────
    'DMF':                      'DMF',
    'Dimethylformamide':        'DMF',
    'dimethylformamide':        'DMF',

    # ── Toluene ──────────────────────────────────────────────────────────────
    'toluene':                  'Toluene',

    # ── DMSO ─────────────────────────────────────────────────────────────────
    'DMSO':                     'DMSO',

    # ── Dioxane ──────────────────────────────────────────────────────────────
    '1,4-dioxane':              'Dioxane',
    'dioxane':                  'Dioxane',

    # ── Chlorobenzene ────────────────────────────────────────────────────────
    'chlorobenzene':            'Chlorobenzene',

    # ── o-Dichlorobenzene ────────────────────────────────────────────────────
    'O-C6H4Cl2':                'o-DCB',
    'orthodichlorobenzene':     'o-DCB',

    # ── TBA (tert-butanol) ───────────────────────────────────────────────────
    # Handled in mixtures below

    # ── None ─────────────────────────────────────────────────────────────────
    '-':                        np.nan,

    # ── MIXTURES ─────────────────────────────────────────────────────────────
    # ACN/TBA
    'methanol, tert-butyl alcohol-acetonitrile':    'MeOH/TBA/ACN',
    'T-BuOH-AN':                                   'TBA/ACN',
    't-BuOH-AN':                                   'TBA/ACN',
    'tert-butanol–acetonitrile':                    'TBA/ACN',
    'tert-butyl alcohol/acetonitrile':              'TBA/ACN',
    'Acetonitrile + tert-butanol':                  'TBA/ACN',
    'acetonitrile and tert-butyl alcohol':          'TBA/ACN',
    'acetonitrile/tert-butanol':                    'TBA/ACN',
    '2-methyl-2-propanol-acetonitrile':             'TBA/ACN',
    'butanol/acetonitrile':                         'TBA/ACN',

    # THF mixtures
    '20%THF-toluene':           'THF/Toluene',
    '20% THF-toluene':          'THF/Toluene',
    'THF/toluene':              'THF/Toluene',
    'THF/DCM (1:1)':            'THF/DCM',
    'DCM/THF (1:1)':            'THF/DCM',

    # DCM mixtures
    'CH2Cl2/MeOH':              'DCM/MeOH',
    'CH2Cl2/MeOH (1:1)':        'DCM/MeOH',
    'MeOH/CH2Cl2 (1:1)':        'DCM/MeOH',
    'CH2Cl2:MeOH':              'DCM/MeOH',
    'CH2Cl2/C2H5OH':            'DCM/Ethanol',
    'CH2Cl2/EtOH':              'DCM/Ethanol',
    'ethanol and dichloromethane (v/v, 4/1)':   'DCM/Ethanol',
    'EtOH/DCM (4 : 1, v/v)':   'DCM/Ethanol',
    'EtOH/DCM':                 'DCM/Ethanol',
    'ethanol:dichloromethane':  'DCM/Ethanol',
    'MeCN-DCM':                 'DCM/ACN',
    'CH3CN-CH2Cl2':             'DCM/ACN',
    'DMF/DCM':                  'DCM/DMF',

    # Chloroform mixtures
    'CH3OH-CHCl3':              'Chloroform/MeOH',
    'CHCl3-CH3OH':              'Chloroform/MeOH',
    'CH3OH/CHCl3':              'Chloroform/MeOH',
    'CH3OH+CHCl3':              'Chloroform/MeOH',
    'CHCl3/MeOH':               'Chloroform/MeOH',
    'CHCl3/CH3OH':              'Chloroform/MeOH',
    'CHCl3/MeOH/EtN3':          'Chloroform/MeOH/TEA',
    'CHCl3/MeOH/Et3N':          'Chloroform/MeOH/TEA',
    'Chloroform + methanol':    'Chloroform/MeOH',
    'Methanol-chloroform':      'Chloroform/MeOH',
    'chloroform/ethanol':       'Chloroform/Ethanol',

    # ACN mixtures
    'Acetonitrile-DMSO':        'ACN/DMSO',
    'CH3CN/C2H5OH':             'ACN/Ethanol',

    # EtOH mixtures
    'EtOH:DMF':                 'Ethanol/DMF',
    'EtOH/DMSO (10:1)':         'Ethanol/DMSO',
    'DMSO:ethanol':             'Ethanol/DMSO',
    'DMF/MeOH':                 'DMF/MeOH',
}


# ── HELPER FUNCTIONS ──────────────────────────────────────────────────────────

def clean_string(s):
    """Strip carriage returns, newlines and extra whitespace."""
    return re.sub(r'[\r\n]+', ' ', str(s)).strip()

def is_smiles(s):
    """Heuristic: long string with multiple SMILES-specific characters."""
    smiles_chars = set('()[]=#@+\\/%')
    return len(s) > 20 and len(smiles_chars & set(s)) >= 3

def normalise_compound(s):
    """Exact match → regex match → SMILES heuristic → return as-is."""
    s = s.strip()

    # 1. Exact match
    if s in COADSORBENT_EXACT:
        return COADSORBENT_EXACT[s]

    # 2. Case-insensitive exact match
    for key, val in COADSORBENT_EXACT.items():
        if s.lower() == key.lower():
            return val

    # 3. Regex match
    for pattern, name in COADSORBENT_MAP:
        if re.search(pattern, s, re.IGNORECASE):
            return name

    # 4. SMILES heuristic
    if is_smiles(s):
        return 'Unknown (SMILES)'

    return s  # return as-is for manual review

def parse_concentration_mM(s):
    """Extract concentration from string and return in mM."""
    match = re.search(r'([\d.]+)\s*(mM|μM|uM|nM|mm|um|nm)', s, re.IGNORECASE)
    if match:
        try:
            value  = float(match.group(1))
            unit   = match.group(2).lower().replace('\u03bc', 'μ')
            factor = UNIT_TO_MM.get(unit, np.nan)
            return value * factor
        except (ValueError, TypeError):
            return np.nan
    return np.nan

def is_non_numeric(v):
    try:
        float(v)
        return False
    except (ValueError, TypeError):
        return True

# ── MAIN PARSING FUNCTION ─────────────────────────────────────────────────────

def parse_co_adsorbent(s):
    """
    Returns (has_coadsorbent: bool, compound_type: str, concentration_mM: float)
    concentration_mM = 0.0 when no co-adsorbent present
    concentration_mM = NaN when present but concentration unknown
    """
    if pd.isna(s) or str(s).strip() in ('-', ''):
        return False, np.nan, 0.0

    s_clean = clean_string(s)

    # Known noise entries
    if any(n.lower() == s_clean.lower() for n in NOISE_ENTRIES):
        return False, np.nan, 0.0

    # Saturated — present but concentration unknown
    if re.match(r'^saturated$', s_clean, re.IGNORECASE):
        return True, 'CDCA (saturated)', np.nan

    # Extract concentration
    conc_mM = parse_concentration_mM(s_clean)

    # Strip concentration prefix to isolate compound name
    name_part = re.sub(r'^[\d.]+\s*(mM|μM|uM|nM|mm|um|nm)\s*',
                       '', s_clean, flags=re.IGNORECASE).strip()
    # Strip "X equiv" prefix
    name_part = re.sub(r'^[\d.]+\s*equiv\s*', '', name_part, flags=re.IGNORECASE).strip()

    compound = normalise_compound(name_part if name_part else s_clean)

    return True, compound, conc_mM

def parse_co_sensitizer(s):
    """
    Returns (has_cosensitizer: bool, name: str)
    """
    if pd.isna(s) or str(s).strip() in ('-', ''):
        return False, np.nan

    s_clean = clean_string(s).strip()

    # Exact match first
    if s_clean in COSENSITIZER_EXACT:
        return True, COSENSITIZER_EXACT[s_clean]

    # Case-insensitive exact match
    for key, val in COSENSITIZER_EXACT.items():
        if s_clean.lower() == key.lower():
            return True, val

    # Strip trailing descriptors
    name = re.sub(r'\s*(cosensitizer|co-sensitizer|ruthenium)\s*',
                  '', s_clean, flags=re.IGNORECASE).strip()

    # Normalise separators (+ → /)
    name = re.sub(r'\s*\+\s*', '/', name)

    return True, name

def parse_absorption_maxima(v):
    try:
        v = float(v)
        return v if 300 <= v <= 900 else np.nan
    except (ValueError, TypeError):
        return np.nan

def normalise_solvent(s):
    if pd.isna(s):
        return np.nan
    s = clean_string(s).strip()

    # Exact match
    if s in SOLVENT_MAP:
        return SOLVENT_MAP[s]

    # Case-insensitive exact match
    for key, val in SOLVENT_MAP.items():
        if s.lower() == key.lower():
            return val

    return s  # return as-is — flag for review


In [37]:
# ── APPLY CO ADBSORBENT ─────────────────────────────────────────────────────────────────────

result = venkatraman['Co-adsorbent'].apply(
    lambda x: pd.Series(parse_co_adsorbent(x),
                        index=['Has_coadsorbent', 'Coadsorbent_type', 'Coadsorbent_conc_mM'])
)
dsscdb_formatted[['Has_coadsorbent', 'Coadsorbent_type', 'Coadsorbent_conc_mM']] = result

# ── DIAGNOSTICS ───────────────────────────────────────────────────────────────

print("=== HAS CO-ADSORBENT ===")
print(dsscdb_formatted['Has_coadsorbent'].value_counts())

print("\n=== COMPOUND TYPE ===")
print(dsscdb_formatted['Coadsorbent_type'].value_counts())

print("\n=== CONCENTRATION (mM) ===")
print(dsscdb_formatted['Coadsorbent_conc_mM'].describe().round(3))

print("\n=== UNMAPPED ENTRIES ===")
known = set(COADSORBENT_EXACT.values()) | \
        set(v for _, v in COADSORBENT_MAP) | \
        {'CDCA (saturated)', 'Unknown (SMILES)'}
unmapped = dsscdb_formatted[
    dsscdb_formatted['Has_coadsorbent'] &
    ~dsscdb_formatted['Coadsorbent_type'].isin(known)
]
print(unmapped['Coadsorbent_type'].value_counts())

=== HAS CO-ADSORBENT ===
Has_coadsorbent
False    3010
True     1416
Name: count, dtype: int64

=== COMPOUND TYPE ===
Coadsorbent_type
CDCA                       1157
DCA                         166
Cholic acid                  43
Isooctylsilane               21
HC-A1                         6
CDCA (saturated)              5
Cholanic acid                 3
4-Guanidinobutyric acid       3
HC-A4                         2
DINHOP                        2
Octanoic acid                 2
1-Decylphosphonic acid        2
PTZ1                          1
PTZ2                          1
Diphenylphosphinic acid       1
3-Phenylpropionic acid        1
Name: count, dtype: int64

=== CONCENTRATION (mM) ===
count    4262.000
mean        2.422
std         8.493
min         0.000
25%         0.000
50%         0.000
75%         0.500
max       160.000
Name: Coadsorbent_conc_mM, dtype: float64

=== UNMAPPED ENTRIES ===
Series([], Name: count, dtype: int64)


In [38]:
# ── APPLY CO SENSITIZER ────────────────────────────────────────────────────────────────────

result = venkatraman['Co-sensitizer'].apply(
    lambda x: pd.Series(parse_co_sensitizer(x),
                        index=['Has_cosensitizer', 'Cosensitizer_name'])
)
dsscdb_formatted[['Has_cosensitizer', 'Cosensitizer_name']] = result

# ── DIAGNOSTICS ───────────────────────────────────────────────────────────────

print("=== HAS CO-SENSITIZER ===")
print(dsscdb_formatted['Has_cosensitizer'].value_counts())

print("\n=== CO-SENSITIZER NAME ===")
print(dsscdb_formatted['Cosensitizer_name'].value_counts())

# ── OVERVIEW ──────────────────────────────────────────────────────────────────
print("=== SHAPE ===")
print(f"Rows: {len(dsscdb_formatted)}")
print(f"Columns: {dsscdb_formatted.shape[1]}")

print("\n=== COLUMNS & DTYPES ===")
print(dsscdb_formatted.dtypes)

print("\n=== COMPLETENESS ===")
completeness = dsscdb_formatted.notna().sum()
pct = (completeness / len(dsscdb_formatted) * 100).round(1)
print(pd.DataFrame({'count': completeness, '%': pct}).to_string())

print("\n=== PERFORMANCE PARAMETERS ===")
print(dsscdb_formatted[['VOC', 'JSC', 'FF', 'PCE']].describe().round(2))

print("\n=== DEVICE DESCRIPTORS ===")
device_cols = ['I2_conc_M', 'Semiconductor_thick', 'Solar_simulator',
               'Dye_loading', 'Exposure_time', 'Active_area',
               'Has_coadsorbent', 'Coadsorbent_conc_mM', 'Has_cosensitizer']
print(dsscdb_formatted[device_cols].describe().round(2))

print("\n=== CATEGORICAL DEVICE FEATURES ===")
for col in ['Coadsorbent_type', 'Solvent_std', 'Semiconductor_type']:
    if col in dsscdb_formatted.columns:
        print(f"\n{col}:")
        print(dsscdb_formatted[col].value_counts().head(10))

print("\n=== MOLECULAR FEATURES ===")
for col in ['SMILES', 'Absorption_max_nm']:
    if col in dsscdb_formatted.columns:
        print(f"\n{col}: {dsscdb_formatted[col].notna().sum()} non-null")

print("\n=== ROWS WITH SMILES ===")
has_smiles = dsscdb_formatted['SMILES'].notna().sum()
print(f"{has_smiles} / {len(dsscdb_formatted)} ({has_smiles/len(dsscdb_formatted)*100:.1f}%)")

print("\n=== CORE COMPLETE (VOC+JSC+FF+PCE+SMILES) ===")
core = ['VOC', 'JSC', 'FF', 'PCE', 'SMILES']
core_complete = dsscdb_formatted[core].notna().all(axis=1).sum()
print(f"{core_complete} / {len(dsscdb_formatted)} ({core_complete/len(dsscdb_formatted)*100:.1f}%)")

=== HAS CO-SENSITIZER ===
Has_cosensitizer
False    4320
True      106
Name: count, dtype: int64

=== CO-SENSITIZER NAME ===
Cosensitizer_name
N719                 14
C1                   12
WS-5                 10
Zn-3                  7
FNE46                 7
Y123                  6
D35                   5
TC2                   4
XW4                   4
TC1                   4
S2                    4
triphenylamine        2
XS-3                  2
IQ21                  2
D149/D131             2
H2LD14                1
bodipy                1
D-205                 1
D35 dye               1
H2LD14/AN-3           1
0.93e-04 mol/cm3      1
0.69e-04 mol/cm3      1
Zinc-porphyrin as     1
0.99e-04 mol/cm3      1
0.73e-04 mol/cm3      1
D131                  1
D149                  1
0.92e-04 mol/cm3      1
0.89e-04 mol/cm3      1
QX20                  1
D3                    1
0.025 mM D35          1
0.025 mM NT35         1
D35/HD2-mono          1
SQ2                   1
MK2 dye          

In [39]:
# ── APPLY FILTERS AND CREATE CLEAN DATAFRAME ──────────────────────────────────
filters = {
    'PCE':  (0.0,  15.21),
    'VOC':  (50,   1200),
    'JSC':  (0.05, 25.0),
    'FF':   (0.30, 0.85),
}

# Drop unused columns first
drop_cols = ['Dye_loading', 'Cosensitizer_name']
clean_dsscdb = dsscdb_formatted.drop(columns=drop_cols)

# Fix units before filtering
clean_dsscdb['VOC'] = clean_dsscdb['VOC'].apply(
    lambda x: x * 1000 if pd.notna(x) and x < 10 else x
)
clean_dsscdb.loc[clean_dsscdb['FF'] > 1, 'FF'] = \
    clean_dsscdb.loc[clean_dsscdb['FF'] > 1, 'FF'] / 100.0

# Apply physical filters — preserve NaN rows
print("Rows removed per filter:")
for col, (low, high) in filters.items():
    before = len(clean_dsscdb)
    mask = clean_dsscdb[col].isna() | clean_dsscdb[col].between(low, high)
    clean_dsscdb = clean_dsscdb[mask].reset_index(drop=True)
    print(f"  {col} [{low}, {high}]: {before - len(clean_dsscdb)} rows removed")

# Clean remaining device outliers
clean_dsscdb.loc[clean_dsscdb['I2_conc_M'] > 0.5, 'I2_conc_M'] = np.nan

print(f"\nBefore filtering: {len(dsscdb_formatted)}")
print(f"After filtering:  {len(clean_dsscdb)}")
print(f"Total removed:    {len(dsscdb_formatted) - len(clean_dsscdb)}")

print(f"\n=== FINAL COLUMN LIST ===")
print(clean_dsscdb.columns.tolist())

print(f"\n=== COMPLETENESS ===")
completeness = clean_dsscdb.notna().sum()
pct = (completeness / len(clean_dsscdb) * 100).round(1)
print(pd.DataFrame({'count': completeness, '%': pct}).to_string())

print(f"\n=== PERFORMANCE PARAMETERS ===")
print(clean_dsscdb[['VOC', 'JSC', 'FF', 'PCE']].describe().round(2))

Rows removed per filter:
  PCE [0.0, 15.21]: 0 rows removed
  VOC [50, 1200]: 18 rows removed
  JSC [0.05, 25.0]: 6 rows removed
  FF [0.3, 0.85]: 14 rows removed

Before filtering: 4426
After filtering:  4388
Total removed:    38

=== FINAL COLUMN LIST ===
['VOC', 'JSC', 'FF', 'PCE', 'SMILES', 'DOI', 'Redox_couple', 'I2_conc_M', 'Semiconductor_type', 'Semiconductor_thick', 'Scatt', 'Solar_simulator', 'Exposure_time', 'Active_area', 'Has_coadsorbent', 'Coadsorbent_type', 'Coadsorbent_conc_mM', 'Has_cosensitizer']

=== COMPLETENESS ===
                     count      %
VOC                   4388  100.0
JSC                   4388  100.0
FF                    4388  100.0
PCE                   4388  100.0
SMILES                4388  100.0
DOI                   4388  100.0
Redox_couple          4344   99.0
I2_conc_M             3231   73.6
Semiconductor_type    4385   99.9
Semiconductor_thick   3773   86.0
Scatt                 1893   43.1
Solar_simulator       4179   95.2
Exposure_time    

In [40]:
def build_device_features(df):
    """
    Encodes device features from the filtered DSSC dataframe.
    Returns a DataFrame with numeric + one-hot encoded columns.
    """

    # ── Columns to exclude (targets, identifiers, metadata) ──
    exclude = ['VOC', 'JSC', 'FF', 'PCE', 'SMILES', 'DOI']

    # ── Numeric columns (keep as-is) ──
    numeric_cols = [
        'I2_conc_M',
        'Semiconductor_thick',
        'Exposure_time',
        'Active_area',
        'Coadsorbent_conc_mM',
        'Scatt',
        'Solar_simulator'
    ]

    # ── Binary columns (already 0/1 or bool) ──
    binary_cols = [
        'Has_coadsorbent',
        'Has_cosensitizer',
    ]

    # ── Categorical columns (need one-hot encoding) ──
    categorical_cols = [
        'Redox_couple',
        'Semiconductor_type',
        'Coadsorbent_type',
    ]

    # Inspect what we're encoding
    print("=== CATEGORICAL VALUE COUNTS ===")
    for col in categorical_cols:
        n_unique = df[col].nunique()
        print(f"\n{col} ({n_unique} unique, {df[col].isna().sum()} missing):")
        print(df[col].value_counts().head(10).to_string())

    # ── Build device DataFrame ──
    parts = []

    # 1. Numeric features
    parts.append(df[numeric_cols].copy())

    # 2. Binary features (ensure int type)
    bin_df = df[binary_cols].copy().astype(float)
    parts.append(bin_df)

    # 3. One-hot encode categoricals
    #    - NaN gets its own implicit encoding (all zeros in that group)
    #    - drop_first=False so that missing = all-zeros is interpretable
    for col in categorical_cols:
        dummies = pd.get_dummies(
            df[col],
            prefix=col,
            drop_first=False,
            dtype=float,
        )
        parts.append(dummies)

    device_df = pd.concat(parts, axis=1)

    print(f"\n=== DEVICE FEATURE SUMMARY ===")
    print(f"Numeric features:     {len(numeric_cols)}")
    print(f"Binary features:      {len(binary_cols)}")
    print(f"Categorical columns:  {len(categorical_cols)}")
    print(f"Total device features: {device_df.shape[1]}")
    print(f"Samples:              {device_df.shape[0]}")

    return device_df

In [41]:
device_df = build_device_features(clean_dsscdb)
device_df

=== CATEGORICAL VALUE COUNTS ===

Redox_couple (9 unique, 44 missing):
Redox_couple
Iodide/Triiodide      3997
Co(II)/Co(III)         245
Spiro-OMeTAD            33
TEMPO/Iodide            30
Bromide/Tribromide      16
DMPIC/DMPIDC            12
Solid_HTM                6
SCN-/SeCN-               3
Cu(I)/Cu(II)             2

Semiconductor_type (6 unique, 3 missing):
Semiconductor_type
TiO2       4273
ZnO          77
NiO          24
zinc          4
Zn2SnO4       4
SnO2          3

Coadsorbent_type (16 unique, 2990 missing):
Coadsorbent_type
CDCA                       1146
DCA                         166
Cholic acid                  43
Isooctylsilane               14
HC-A1                         6
CDCA (saturated)              5
Cholanic acid                 3
4-Guanidinobutyric acid       3
HC-A4                         2
DINHOP                        2

=== DEVICE FEATURE SUMMARY ===
Numeric features:     7
Binary features:      2
Categorical columns:  3
Total device features: 40
Sam

,I2_conc_M,Semiconductor_thick,Exposure_time,Active_area,Coadsorbent_conc_mM,Scatt,Solar_simulator,Has_coadsorbent,Has_cosensitizer,Redox_couple_Bromide/Tribromide,...,Coadsorbent_type_Cholic acid,Coadsorbent_type_DCA,Coadsorbent_type_DINHOP,Coadsorbent_type_Diphenylphosphinic acid,Coadsorbent_type_HC-A1,Coadsorbent_type_HC-A4,Coadsorbent_type_Isooctylsilane,Coadsorbent_type_Octanoic acid,Coadsorbent_type_PTZ1,Coadsorbent_type_PTZ2
0,0.03,NaN,15.0,NaN,0.0,NaN,100.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.00,8.0,5.0,0.158,0.0,5.0,100.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.00,8.0,5.0,0.158,0.0,5.0,100.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.00,8.0,5.0,0.158,0.0,5.0,100.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,NaN,8.0,5.0,0.158,0.0,5.0,100.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4383,0.02,14.0,24.0,0.120,0.0,NaN,100.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4384,0.02,14.0,24.0,0.120,0.0,NaN,100.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4385,0.02,14.0,24.0,0.120,0.0,NaN,100.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4386,0.02,14.0,24.0,0.120,0.0,NaN,100.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [42]:
import cheminfo_functions as cheminfo
morgan_df = cheminfo.morgan_fp(clean_dsscdb, nBits=1024, radius=3, smilesCol="SMILES")

In [43]:
entire_df = pd.concat([morgan_df, device_df, clean_dsscdb['PCE']], axis=1, join='inner')

dupes = entire_df[entire_df.duplicated(keep=False)]
dupe_groups = (
    dupes
    .groupby(list(entire_df.columns), dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

C:\Users\anamj\AppData\Local\Temp\ipykernel_8896\711800031.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dupes
C:\Users\anamj\AppData\Local\Temp\ipykernel_8896\711800031.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dupes
C:\Users\anamj\AppData\Local\Temp\ipykernel_8896\711800031.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragme

In [45]:
dupe_groups

,Bit_0,Bit_1,Bit_2,Bit_3,Bit_4,Bit_5,Bit_6,Bit_7,Bit_8,Bit_9,...,Coadsorbent_type_DINHOP,Coadsorbent_type_Diphenylphosphinic acid,Coadsorbent_type_HC-A1,Coadsorbent_type_HC-A4,Coadsorbent_type_Isooctylsilane,Coadsorbent_type_Octanoic acid,Coadsorbent_type_PTZ1,Coadsorbent_type_PTZ2,PCE,count
0,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.32,2
19,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.40,2
21,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.88,2
22,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.20,2
23,0,0,0,0,0,0,0,0,0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.20,2
24,0,0,0,0,0,0,0,1,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.50,2
25,0,0,0,1,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.47,2
26,0,0,0,1,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.93,2
27,0,0,0,1,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.19,2
28,0,0,0,1,0,0,0,1,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.10,2


In [46]:
print(f"Before: {len(entire_df)} rows")
print(f"Duplicate rows: {dupes.shape[0]}")
print(f"Unique duplicate groups: {len(dupe_groups)}")

entire_df_clean = entire_df.drop_duplicates(keep='first')
print(f"After: {len(entire_df_clean)} rows")
print(f"Removed: {len(entire_df) - len(entire_df_clean)} rows")

Before: 4388 rows
Duplicate rows: 74
Unique duplicate groups: 37
After: 4351 rows
Removed: 37 rows


In [47]:
entire_df_clean.to_csv('Data/dye_device_pce_without_dupes_new.csv')